# ⚽ World Cup 2026 Win Probability — Phase 2: Transfer Learning

**Goal:** Take the pre-trained NBA WinProbNet, adapt it for 3-class football outcomes,
and fine-tune on the 9,682 football snapshots built in Phase 1.

## Transfer learning strategy
```
NBA model (12→128→64→32→1, sigmoid)
         ↓
Step 1:  Replace output layer  32→1  →  32→3  (softmax, 3 classes)
Step 2:  Expand input layer    12→128  →  14→128
         - Cols 0–11: copy NBA weights (same game-state features)
         - Cols 12–13: small random init (is_neutral_venue, score_state)
Step 3:  Stage 1 — freeze hidden layers, train only input+output (10 epochs)
Step 4:  Stage 2 — unfreeze all, fine-tune at LR=1e-5 (50 epochs max)
```

## Feature mapping NBA → Football
| Position | NBA feature | Football feature |
|---|---|---|
| 0 | score_diff | goal_diff |
| 1 | time_remaining_sec | time_remaining_sec |
| 2 | quarter (1-4) | half (1-2) |
| 3 | quarter_time_elapsed_pct | match_time_pct |
| 4 | home_net_rtg | home_rank_norm |
| 5 | away_net_rtg | away_rank_norm |
| 6 | net_rtg_diff | rank_diff |
| 7 | home_series_wins | home_group_pts |
| 8 | away_series_wins | away_group_pts |
| 9 | is_playoffs | is_knockout |
| 10 | is_overtime | is_extra_time |
| 11 | lead_changes_norm | lead_changes_norm |
| 12 | — | is_neutral_venue ← NEW |
| 13 | — | score_state ← NEW |


## Cell 1 — Environment & GPU

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd
import pickle, json, warnings, time
from pathlib import Path
from copy import deepcopy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    brier_score_loss, log_loss
)
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Paths
BASE_DIR     = Path('world_cup_win_prob')
NBA_DIR      = Path('../nba_win_prob/model')  # adjust if repo layout differs
PROC_DIR     = BASE_DIR / 'processed'
MODEL_DIR    = BASE_DIR / 'model'
PLOT_DIR     = BASE_DIR / 'plots' / 'phase2'
for d in [MODEL_DIR, PLOT_DIR]: d.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), '❌ CUDA not found — use GPU runtime'
DEVICE = torch.device('cuda')
torch.manual_seed(42); np.random.seed(42)
torch.backends.cudnn.benchmark = True

plt.style.use('dark_background')
DARK_BG = '#0d1117'; CARD_BG = '#161b22'; BORDER = '#30363d'

print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'CUDA   : {torch.version.cuda}  |  PyTorch: {torch.__version__}')
print(f'Model  → {MODEL_DIR.resolve()}')


AssertionError: ❌ CUDA not found — use GPU runtime

## Cell 2 — Load Phase 1 Dataset

In [ ]:
df = pd.read_parquet(PROC_DIR / 'features_raw.parquet')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

# Feature ordering aligned to NBA model weights for maximum transfer
# Positions 0-11 map directly to NBA features; 12-13 are new
FEATURE_COLS = [
    'goal_diff',           # 0  ↔ score_diff
    'time_remaining_sec',  # 1  ↔ time_remaining_sec
    'half',                # 2  ↔ quarter
    'match_time_pct',      # 3  ↔ quarter_time_elapsed_pct
    'home_rank_norm',      # 4  ↔ home_net_rtg
    'away_rank_norm',      # 5  ↔ away_net_rtg
    'rank_diff',           # 6  ↔ net_rtg_diff
    'home_group_pts',      # 7  ↔ home_series_wins
    'away_group_pts',      # 8  ↔ away_series_wins
    'is_knockout',         # 9  ↔ is_playoffs
    'is_extra_time',       # 10 ↔ is_overtime
    'lead_changes_norm',   # 11 ↔ lead_changes_norm
    'is_neutral_venue',    # 12  NEW
    'score_state',         # 13  NEW
]
TARGET_COL = 'outcome'  # 0=away win, 1=draw, 2=home win
N_FEATURES  = len(FEATURE_COLS)   # 14
N_CLASSES   = 3

# Verify all features exist
missing = [c for c in FEATURE_COLS if c not in df.columns]
assert not missing, f'Missing features: {missing}'

X      = df[FEATURE_COLS].values.astype(np.float32)
y      = df[TARGET_COL].values.astype(np.int64)
groups = df['match_id'].values

print(f'\nFeatures : {N_FEATURES}')
print(f'Samples  : {len(X):,}')
print(f'Classes  : {N_CLASSES}  (0=away win, 1=draw, 2=home win)')
print(f'\nOutcome distribution:')
for label, c in [('Away win (0)',0),('Draw (1)',1),('Home win (2)',2)]:
    n = (y==c).sum()
    print(f'  {label}: {n:,} ({n/len(y):.1%})')
print(f'\nUnique matches: {np.unique(groups).shape[0]:,}')


## Cell 3 — Football Model Architecture

In [ ]:
class FootballWinProbNet(nn.Module):
    """
    3-class win probability model for football.
    Same hidden layer structure as NBA WinProbNet (128→64→32).
    Output: 3 logits (away win / draw / home win).
    Uses CrossEntropyLoss during training (expects raw logits).
    """
    def __init__(self, n_features=14, hidden_dims=None,
                 dropout=0.30, use_batchnorm=True, n_classes=3):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 64, 32]
        self.n_features  = n_features
        self.hidden_dims = hidden_dims
        self.n_classes   = n_classes

        layers, in_dim = [], n_features
        for out_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, out_dim, bias=not use_batchnorm))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(out_dim))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(p=dropout))
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, n_classes))  # 3-class output
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        """Returns raw logits. Apply softmax externally for probabilities."""
        return self.net(x)

    def predict_proba(self, x):
        """Returns softmax probabilities for all 3 classes."""
        with torch.no_grad():
            return F.softmax(self.net(x), dim=1)


# Verify shapes
_test = FootballWinProbNet()
_x    = torch.randn(4, 14)
_out  = _test(_x)
print(f'Architecture: 14 → {" → ".join(map(str, [128,64,32]))} → 3')
print(f'Output shape : {tuple(_out.shape)}  (batch=4, classes=3)')
print(f'Params       : {sum(p.numel() for p in _test.parameters()):,}')
print()
# Layer index mapping for weight transfer
print('Layer indices (for weight transfer):')
for i, layer in enumerate(_test.net):
    print(f'  net.{i}: {layer}')


## Cell 4 — Transfer NBA Weights

In [ ]:
# ── Load NBA model ─────────────────────────────────────────────────────────
# Try common locations for the NBA model file
nba_paths = [
    NBA_DIR / 'win_prob_net.pth',
    Path('"C:\\Users\\Admin\\OneDrive\\Documents\\Schoolwork\\Projects\\NBA-Live-Win-Probability\\nba_win_prob\\model\\win_prob_net.pth"'),
    Path('../../NBA-Live-Win-Probability/nba_win_prob/model/win_prob_net.pth'),
]
nba_path = next((p for p in nba_paths if p.exists()), None)
assert nba_path, (
    f'NBA model not found. Copy win_prob_net.pth to {NBA_DIR} '
    f'or update NBA_DIR in Cell 1.'
)
print(f'Loading NBA model from: {nba_path}')

nba_ckpt  = torch.load(nba_path, map_location='cpu', weights_only=False)
nba_state = nba_ckpt['model_state_dict']
nba_cfg   = nba_ckpt['model_config']

print(f'NBA architecture: {nba_cfg["n_features"]} → '
      f'{nba_cfg["hidden_dims"]} → 1')
print(f'NBA version: {nba_ckpt.get("model_version", "phase2")}')
print(f'NBA AUC: {nba_ckpt["test_metrics"]["roc_auc"]:.4f}')

# ── Create football model ──────────────────────────────────────────────────
football_model = FootballWinProbNet(
    n_features=N_FEATURES,
    hidden_dims=nba_cfg['hidden_dims'],   # [128, 64, 32] — same as NBA
    dropout=nba_cfg['dropout'],
    use_batchnorm=nba_cfg['use_batchnorm'],
    n_classes=N_CLASSES,
).to(DEVICE)

# ── Transfer hidden layers (layers 1-3: 128→64→32) ────────────────────────
# These layers learned general game-state representations from 962K NBA plays.
# Layer index mapping (both models use same hidden architecture):
#   net.0  Linear(in,128)   ← PARTIAL transfer (cols 0-11 from NBA)
#   net.1  BatchNorm(128)   ← FULL copy
#   net.3  Linear(128,64)   ← FULL copy
#   net.4  BatchNorm(64)    ← FULL copy
#   net.6  Linear(64,32)    ← FULL copy
#   net.7  BatchNorm(32)    ← FULL copy
#   net.9  Linear(32,3)     ← SKIP (new 3-class output)

football_state = football_model.state_dict()
transferred = []; skipped = []

for nba_key, nba_val in nba_state.items():
    if 'net.12' in nba_key or 'net.9' in nba_key:
        # Skip NBA output layer (was 32→1, now 32→3)
        skipped.append(nba_key)
        continue

    if nba_key not in football_state:
        skipped.append(nba_key)
        continue

    if nba_key == 'net.0.weight':
        # Input layer: NBA is [128,12], football needs [128,14]
        # Copy first 12 columns (matching features), leave cols 12-13 as init
        new_weight = football_state['net.0.weight'].clone()
        new_weight[:, :12] = nba_val          # copy overlapping features
        new_weight[:, 12:] *= 0.01            # shrink new-feature cols to near-zero
        football_state['net.0.weight'] = new_weight
        transferred.append(f'{nba_key}  [partial: cols 0-11 of 14]')

    elif football_state[nba_key].shape == nba_val.shape:
        football_state[nba_key] = nba_val
        transferred.append(nba_key)
    else:
        skipped.append(f'{nba_key} (shape mismatch)')

football_model.load_state_dict(football_state)

print(f'\n✅ Transfer complete')
print(f'   Transferred : {len(transferred)} parameter tensors')
print(f'   Skipped     : {len(skipped)} (new input cols + output layer)')
print(f'\nTransferred layers:')
for t in transferred:
    print(f'  {t}')
print(f'\nSkipped (expected):')
for s in skipped:
    print(f'  {s}')


## Cell 5 — Split, Scale, Push to GPU

In [ ]:
# Group-aware split — all snapshots from one match stay in the same split
gss_outer = GroupShuffleSplit(1, test_size=0.20, random_state=42)
tv_idx, te_idx = next(gss_outer.split(X, y, groups))

gss_inner = GroupShuffleSplit(1, test_size=0.25, random_state=42)
tr_idx, va_idx = next(gss_inner.split(
    X[tv_idx], y[tv_idx], groups[tv_idx]))
tr_idx, va_idx = tv_idx[tr_idx], tv_idx[va_idx]

# Verify no match leaks across splits
assert not (set(groups[tr_idx]) & set(groups[va_idx]))
assert not (set(groups[tr_idx]) & set(groups[te_idx]))

# Fit scaler on train only
scaler = StandardScaler()
X_train = scaler.fit_transform(X[tr_idx]).astype(np.float32)
X_val   = scaler.transform(X[va_idx]).astype(np.float32)
X_test  = scaler.transform(X[te_idx]).astype(np.float32)
y_train, y_val, y_test = y[tr_idx], y[va_idx], y[te_idx]

# Save scaler immediately
with open(MODEL_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Push to GPU
def to_gpu_f(a): return torch.from_numpy(a).to(DEVICE)
def to_gpu_l(a): return torch.from_numpy(a).long().to(DEVICE)

X_tr_g, y_tr_g = to_gpu_f(X_train), to_gpu_l(y_train)
X_va_g, y_va_g = to_gpu_f(X_val),   to_gpu_l(y_val)
X_te_g, y_te_g = to_gpu_f(X_test),  to_gpu_l(y_test)

BATCH = 256  # small batch for small dataset
train_loader = DataLoader(
    TensorDataset(X_tr_g, y_tr_g),
    batch_size=BATCH, shuffle=True, drop_last=False)
val_loader = DataLoader(
    TensorDataset(X_va_g, y_va_g),
    batch_size=BATCH*4, shuffle=False)

print(f'Train : {len(tr_idx):,} samples  ({np.unique(groups[tr_idx]).shape[0]} matches)')
print(f'Val   : {len(va_idx):,} samples  ({np.unique(groups[va_idx]).shape[0]} matches)')
print(f'Test  : {len(te_idx):,} samples  ({np.unique(groups[te_idx]).shape[0]} matches)')
print(f'Batch : {BATCH}  |  Batches/epoch: {len(train_loader)}')
print(f'VRAM  : {torch.cuda.memory_allocated()/1e9:.3f} GB')


## Cell 6 — Stage 1: Train Input + Output Only (Frozen Hidden)

In [ ]:
# ── Freeze all hidden layers ──────────────────────────────────────────────
# Only the input projection (net.0) and output head (last Linear) are trainable.
# This lets the new football features adapt WITHOUT disturbing the
# pre-trained game-state representations in layers 1-3.

for name, param in football_model.named_parameters():
    # Freeze middle layers: BatchNorm and Linear for hidden dims
    is_hidden = any(f'net.{i}' in name for i in [1,2,3,4,5,6,7,8,9,10,11])
    param.requires_grad = not is_hidden

trainable = sum(p.numel() for p in football_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in football_model.parameters())
print(f'Stage 1 — trainable: {trainable:,} / {total:,} params '
      f'({trainable/total:.1%})')

criterion  = nn.CrossEntropyLoss()
optimizer1 = optim.AdamW(
    [p for p in football_model.parameters() if p.requires_grad],
    lr=3e-4, weight_decay=1e-4)


@torch.no_grad()
def evaluate(loader):
    football_model.eval()
    total_loss = 0.0; all_p, all_l = [], []
    for xb, yb in loader:
        logits = football_model(xb)
        total_loss += criterion(logits, yb).item() * len(xb)
        all_p.append(F.softmax(logits, dim=1).cpu().numpy())
        all_l.append(yb.cpu().numpy())
    probs  = np.vstack(all_p)
    labels = np.concatenate(all_l)
    loss   = total_loss / len(loader.dataset)
    acc    = (probs.argmax(1) == labels).mean()
    return loss, probs, labels, acc


S1_EPOCHS = 15
hist1     = []
print(f'Stage 1: {S1_EPOCHS} epochs (input + output layers only)\n')

for epoch in range(1, S1_EPOCHS+1):
    football_model.train()
    ep_loss = 0.0
    for xb, yb in train_loader:
        optimizer1.zero_grad(set_to_none=True)
        loss = criterion(football_model(xb), yb)
        loss.backward()
        optimizer1.step()
        ep_loss += loss.item() * len(xb)
    tr_loss = ep_loss / len(train_loader.dataset)

    val_loss, _, _, val_acc = evaluate(val_loader)
    hist1.append({'epoch':epoch,'tr_loss':tr_loss,
                  'val_loss':val_loss,'val_acc':val_acc})

    if epoch % 5 == 0 or epoch == 1:
        print(f'  Ep {epoch:2d}  tr={tr_loss:.4f}  '
              f'val={val_loss:.4f}  acc={val_acc:.3f}')

print(f'\nStage 1 complete.')


## Cell 7 — Stage 2: Full Fine-Tune (All Layers, Low LR)

In [ ]:
# ── Unfreeze all layers ────────────────────────────────────────────────────
for param in football_model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in football_model.parameters() if p.requires_grad)
print(f'Stage 2 — all {trainable:,} params trainable')
print(f'Learning rate: 1e-5  (10× lower than Stage 1 to protect transferred weights)')

optimizer2 = optim.AdamW(
    football_model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=20, eta_min=1e-7)

S2_EPOCHS     = 60
PATIENCE      = 10
MIN_DELTA     = 1e-5
best_val_loss = float('inf')
best_state    = None
patience_ctr  = 0
hist2         = []

print(f'\nStage 2: up to {S2_EPOCHS} epochs  patience={PATIENCE}\n')

t0 = time.time()
for epoch in range(1, S2_EPOCHS+1):
    football_model.train()
    ep_loss = 0.0
    for xb, yb in train_loader:
        optimizer2.zero_grad(set_to_none=True)
        loss = criterion(football_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(football_model.parameters(), 1.0)
        optimizer2.step()
        ep_loss += loss.item() * len(xb)
    tr_loss = ep_loss / len(train_loader.dataset)

    val_loss, _, _, val_acc = evaluate(val_loader)
    scheduler.step()
    hist2.append({'epoch':epoch,'tr_loss':tr_loss,
                  'val_loss':val_loss,'val_acc':val_acc})

    improved = val_loss < best_val_loss - MIN_DELTA
    if improved:
        best_val_loss = val_loss
        best_state    = deepcopy(football_model.state_dict())
        patience_ctr  = 0
        tag = ' ★'
    else:
        patience_ctr += 1
        tag = ''

    if epoch % 5 == 0 or epoch == 1 or improved:
        print(f'  Ep {epoch:2d}  tr={tr_loss:.4f}  '
              f'val={val_loss:.4f}  acc={val_acc:.3f}  '
              f'[{time.time()-t0:.0f}s]{tag}')

    if patience_ctr >= PATIENCE:
        print(f'\n⏹  Early stop at epoch {epoch}')
        break

football_model.load_state_dict(best_state)
best_ep = min(range(len(hist2)), key=lambda i: hist2[i]['val_loss']) + 1
print(f'\n✅ Stage 2 complete  ({time.time()-t0:.0f}s)  best epoch: {best_ep}')


## Cell 8 — Test Set Evaluation

In [ ]:
_, te_probs, te_labels, te_acc = evaluate(
    DataLoader(TensorDataset(X_te_g, y_te_g),
               batch_size=512, shuffle=False))

te_preds = te_probs.argmax(1)
class_names = ['Away win', 'Draw', 'Home win']

print('='*62)
print('TEST SET EVALUATION')
print('='*62)
print(f'Overall accuracy: {te_acc:.4f}')
print(f'Log loss        : {log_loss(te_labels, te_probs):.4f}')
print()

# Per-class Brier scores (one-vs-rest)
print('Per-class Brier score (one-vs-rest):')
for c, name in enumerate(class_names):
    b = brier_score_loss((te_labels==c).astype(int), te_probs[:,c])
    print(f'  {name:<12}: {b:.4f}')
print()
print(classification_report(te_labels, te_preds,
      target_names=class_names, digits=3))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7,6), facecolor=DARK_BG)
cm = confusion_matrix(te_labels, te_preds, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names, ax=ax,
    linewidths=0.5, cbar_kws={'shrink':0.8})
ax.set_xlabel('Predicted', color='#e6edf3')
ax.set_ylabel('Actual', color='#e6edf3')
ax.set_title('Confusion Matrix (normalised)', color='#e6edf3', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR/'confusion_matrix.png', dpi=150, bbox_inches='tight',
            facecolor=DARK_BG)
plt.show()

# Learning curves
hist_all  = [{'stage':1,**h} for h in hist1] + \
            [{'stage':2,'epoch':h['epoch']+len(hist1),**h} for h in hist2]
hist_df   = pd.DataFrame(hist_all)

fig, axes = plt.subplots(1,2, figsize=(16,5), facecolor=DARK_BG)
for ax, col, label in [
    (axes[0],'val_loss','Validation Loss'),
    (axes[1],'val_acc', 'Validation Accuracy'),
]:
    ax.set_facecolor(CARD_BG)
    ax.plot(hist_df.index, hist_df[col], color='steelblue', lw=1.8)
    ax.axvline(len(hist1), color='gold', lw=1, ls='--', label='S1→S2')
    ax.set_title(label, color='#e6edf3', fontweight='bold')
    ax.grid(alpha=0.2); ax.legend(fontsize=8)
plt.suptitle('Training History (Stage 1 + Stage 2)', color='#e6edf3', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR/'learning_curves.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


## Cell 9 — 3-Class Temperature Scaling

In [ ]:
# Temperature scaling for multi-class calibration.
# A single scalar T is applied to all logits before softmax.
# Optimise T on the validation set to minimise cross-entropy (log loss).

football_model.eval()
with torch.no_grad():
    logits_val = football_model(X_va_g).cpu().numpy()  # shape [N, 3]

def nll_3class(T):
    """Negative log likelihood after temperature scaling."""
    scaled  = logits_val / T
    # Subtract max for numerical stability before softmax
    scaled -= scaled.max(1, keepdims=True)
    probs   = np.exp(scaled)
    probs  /= probs.sum(1, keepdims=True)
    probs   = np.clip(probs, 1e-7, 1-1e-7)
    return log_loss(y_val, probs)

result = minimize_scalar(nll_3class, bounds=(0.1, 10.0),
                         method='bounded', options={'xatol':1e-6})
T_opt  = float(result.x)

# Verify calibration improved
raw_nll  = nll_3class(1.0)
cal_nll  = nll_3class(T_opt)

print(f'Temperature T  : {T_opt:.6f}')
print(f'Log loss raw   : {raw_nll:.5f}')
print(f'Log loss cal   : {cal_nll:.5f}  '
      f'{"(improved ✅)" if cal_nll < raw_nll else "(no improvement)"}')

# Save temperature
temp_data = {
    'temperature'  : T_opt,
    'model_version': 'phase2_football_transfer',
    'n_classes'    : 3,
    'nll_before'   : raw_nll,
    'nll_after'    : cal_nll,
}
with open(MODEL_DIR / 'temperature.json', 'w') as f:
    json.dump(temp_data, f, indent=2)
print(f'✅ Saved temperature.json')

# Reliability diagram per class
from sklearn.calibration import calibration_curve
scaled_l = logits_val / T_opt
scaled_l -= scaled_l.max(1, keepdims=True)
cal_probs = np.exp(scaled_l); cal_probs /= cal_probs.sum(1, keepdims=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=DARK_BG)
for c, (name, ax) in enumerate(zip(class_names, axes)):
    y_bin  = (y_val == c).astype(int)
    frac_r, mean_r = calibration_curve(y_bin, te_probs[:,c] if c<te_probs.shape[1] else cal_probs[:,c], n_bins=8)
    frac_c, mean_c = calibration_curve(y_bin, cal_probs[:, c], n_bins=8)
    ax.set_facecolor(CARD_BG)
    ax.plot([0,1],[0,1],'--',color='#8b949e',lw=1,label='Perfect')
    ax.plot(mean_r, frac_r, 'o-', color='tomato',  lw=1.5, label='Before T')
    ax.plot(mean_c, frac_c, 's-', color='steelblue',lw=1.5, label='After T')
    ax.set_title(name, color='#e6edf3', fontweight='bold')
    ax.set_xlabel('Mean predicted prob'); ax.set_ylabel('Fraction positive')
    ax.legend(fontsize=8); ax.grid(alpha=0.2)
plt.suptitle('Calibration: Before vs After Temperature Scaling',
             color='#e6edf3', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_DIR/'calibration.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


## Cell 10 — Save Model Artifacts

In [ ]:
# Final test evaluation WITH temperature scaling
football_model.eval()
with torch.no_grad():
    te_logits = football_model(X_te_g).cpu().numpy()
te_logits_t  = te_logits / T_opt
te_logits_t -= te_logits_t.max(1, keepdims=True)
te_probs_cal = np.exp(te_logits_t)
te_probs_cal/= te_probs_cal.sum(1, keepdims=True)
te_preds_cal  = te_probs_cal.argmax(1)
te_acc_cal    = (te_preds_cal == te_labels).mean()

print('Test accuracy (with temperature scaling):', f'{te_acc_cal:.4f}')
print('Per-class Brier (calibrated):')
for c, name in enumerate(class_names):
    b = brier_score_loss((te_labels==c).astype(int), te_probs_cal[:,c])
    print(f'  {name:<12}: {b:.4f}')

# Save model
model_path = MODEL_DIR / 'win_prob_net.pth'
torch.save({
    'model_state_dict': football_model.state_dict(),
    'model_config': {
        'n_features'   : N_FEATURES,
        'hidden_dims'  : [128, 64, 32],
        'dropout'      : 0.30,
        'use_batchnorm': True,
        'n_classes'    : N_CLASSES,
    },
    'feature_cols'   : FEATURE_COLS,
    'model_version'  : 'phase2_football_transfer',
    'nba_source'     : str(nba_path),
    'test_metrics'   : {
        'accuracy'  : float(te_acc_cal),
        'log_loss'  : float(log_loss(te_labels, te_probs_cal)),
        'brier_away': float(brier_score_loss((te_labels==0).astype(int), te_probs_cal[:,0])),
        'brier_draw': float(brier_score_loss((te_labels==1).astype(int), te_probs_cal[:,1])),
        'brier_home': float(brier_score_loss((te_labels==2).astype(int), te_probs_cal[:,2])),
    },
}, model_path)

print(f'\n✅ Files saved to {MODEL_DIR}:')
for f in MODEL_DIR.iterdir():
    print(f'   {f.name}  ({f.stat().st_size/1024:.1f} KB)')

print(f"""
\n{'='*60}
PHASE 2 COMPLETE
{'='*60}
Model    : 14 → 128 → 64 → 32 → 3  (softmax, 3-class)
Transfer : NBA hidden layers (128→64→32) fully transferred
           Input layer partially transferred (cols 0-11 of 14)
           Output layer trained from scratch (32→3)
Accuracy : {te_acc_cal:.4f}
Temp T   : {T_opt:.4f}

Next → phase3_live_dashboard.ipynb
  Wire worldcup26.ir live feed into Streamlit app
  Adapt NBA dashboard for 3-class probability display
""")
